In [1]:
import json
import re
from collections import Counter

# --------- Paths ----------
IN_PATH  = "./newData.jsonl"
OUT_PATH = "./allData/HebNLI_train_new_data.clean.jsonl"

# --------- Columns ----------
TEXT1 = "translation1"
TEXT2 = "translation2"
LABEL = "original_label"
GENRE = "genre"

ALLOWED_LABELS = {"entailment", "contradiction", "neutral"}

# --------- Regex ----------
HEBREW_RE = re.compile(r"[\u0590-\u05FF]")          # any Hebrew char
NIQQUD_RE = re.compile(r"[\u0591-\u05C7]")          # niqqud + cantillation
LATIN_RE  = re.compile(r"[A-Za-z]")                 # English letters
# Keep only Hebrew, digits, spaces, and common punctuation
NON_ALLOWED_RE = re.compile(r"[^\u0590-\u05FF0-9\s\.\,\?\!\:\;\-\(\)\"\'\/]")

WHITESPACE_RE = re.compile(r"\s+")

def clean_hebrew_text(s: str) -> str:
    if not isinstance(s, str):
        return ""

    # Remove direction marks (common in Hebrew text)
    s = s.replace("\u200f", " ").replace("\u200e", " ")  # RLM/LRM

    # Remove niqqud
    s = NIQQUD_RE.sub("", s)

    # Remove English letters
    s = LATIN_RE.sub("", s)

    # Remove unwanted chars but keep punctuation above
    s = NON_ALLOWED_RE.sub(" ", s)

    # Collapse multiple whitespace into one
    s = WHITESPACE_RE.sub(" ", s).strip()

    return s

def is_valid_sentence(s: str) -> bool:
    # Must contain at least 2 Hebrew characters after cleaning
    if not s:
        return False
    heb_count = len(re.findall(r"[\u0590-\u05FF]", s))
    return heb_count >= 2

# --------- Streaming clean ----------
seen_pairs = set()
stats = Counter()

with open(IN_PATH, "r", encoding="utf-8") as fin, open(OUT_PATH, "w", encoding="utf-8") as fout:
    for line in fin:
        line = line.strip()
        if not line:
            stats["skip_empty_line"] += 1
            continue

        try:
            ex = json.loads(line)
        except Exception:
            stats["skip_bad_json"] += 1
            continue

        stats["rows_in"] += 1

        # label validation
        lab = str(ex.get(LABEL, "")).strip().lower()
        if lab not in ALLOWED_LABELS:
            stats["skip_bad_label"] += 1
            continue

        t1 = clean_hebrew_text(ex.get(TEXT1, ""))
        t2 = clean_hebrew_text(ex.get(TEXT2, ""))

        # empty / junk sentences
        if not is_valid_sentence(t1) or not is_valid_sentence(t2):
            stats["skip_empty_or_nonhebrew_sentence"] += 1
            continue

        # deduplicate on cleaned pair
        key = (t1, t2)
        if key in seen_pairs:
            stats["skip_duplicate_pair"] += 1
            continue
        seen_pairs.add(key)

        # write cleaned record (keep only needed fields + optional pairID)
        out = {
            TEXT1: t1,
            TEXT2: t2,
            LABEL: lab,
            GENRE: str(ex.get(GENRE, "")).strip()
            
        }
        if "pairID" in ex:
            out["pairID"] = ex["pairID"]

        fout.write(json.dumps(out, ensure_ascii=False) + "\n")
        stats["rows_out"] += 1

print("---- Cleaning summary ----")
for k in [
    "rows_in",
    "rows_out",
    "skip_empty_line",
    "skip_bad_json",
    "skip_bad_label",
    "skip_empty_or_nonhebrew_sentence",
    "skip_duplicate_pair",
]:
    print(f"{k}: {stats.get(k,0)}")

print("Output file:", OUT_PATH)


---- Cleaning summary ----
rows_in: 144
rows_out: 144
skip_empty_line: 0
skip_bad_json: 6
skip_bad_label: 0
skip_empty_or_nonhebrew_sentence: 0
skip_duplicate_pair: 0
Output file: ./allData/HebNLI_train_new_data.clean.jsonl
